Ensemble Models 
- Making the models better should have different criteria 
- Same model, different features (explore)
- All access same data
- Find specific strengths

1. Bagging (bootstrapping - take original data and split up into smaller datasets, so each model gets a different set of data), 1 model for each dataset, trained in parallel
    - If it is binary, which is the majority class
    - Regression: mathematical average
    - Classification: which one is the majority class
    - Probability of each class --> average --> highest probability assigned
    - Decision tree is apart of this
    - Out of Bag Error --> OOB, train on bootstrap (2/3) of the data
    - out of bag (1/3)
    - out of bag error --> observation --> predict on trees that didn't have in bootstrap --> aggregate those predictions --> OOB error
    - The model will be helped find that training 
    - Bootstrap sampling: generates diverse subsets for training base learners --> Diverse base learners: are trained on sampled subsets of the data --> Final prediction of the ensemble is reached by model aggregation 

Decision Trees 
- easy to follow (visualize)
- explainability
- good with outliers 
- any tree based model, you don't need to scale data
- Disadvantages: overfit (memorizes data), small change to the data can lead to a big change in the model, cannot predict outside training range, if there are who depths to the data it will not do well in predicting

Bias vs. Variance 
- Bias goes together with underfitting, oversimplifies the data and doesn't capture the true depth of the relationships, misses relationships that are predicted, training error goes down
- Variance is the sensitivity to the data, overfitting, really good on the training data, poor on the testing data, its memorized the data 
- The best spot is where bias and variance overlap 
- irreducable error: the error you cannot get rid of no matter how well it tests on the testing dataset; it is the baseline noise in the data, missing a feature that is highly predictive 
- natural variation --> measurement error --> missing features: stuff you cannot control

Bagging Tree Ensembles 
- Each tree uses all features in its bagged dataset at each split in the tree
- default = use all data in bootstrap for each node (can change)
- optimize split threshold 
- Bagging classifier: can use default DecisionTrees or other base model 
- Bagging Regressor: can use default DecisionTree or other base model 
- averages across models 
- trains each model in depth
- can parallelize 
- bootstrap overlap substantially, weak on inbalanced data

Random Forest Models
- randomly picks a different set of features 
- standard decision tree: all features are available from which to select the best split 
- random decision tree: only some randomly selected subsets of features are available from which to select the best split
- splits are optimized, there are many weak learners "different expertise"
- balance of speed + prediction 
- decorrelated trees
- feature_importances 
- strong deafult model and works well without any hyperparameter tuning 
- but it does have class imbalances and it is slow on big data

Extra Random Tree Models 
- they are fast to run
- good on noisy data
- there is higher bias = meaning underfitting data

Random Forest Parameters
- n_estimators=100 (tree stability)
- max_features= 'sqrt' (# features in each split)
- max_depth=None (# levels - None, 10, 20, 30)
- max_leaf_nodes=None (# terminal points)
- min_samples_leaf=1
- min_samples_split=2 (# samples before split 2, 5, 10)
- random_state = 342, 42 (provides predictability with how the data is split up) 

2. Boosting (sequential training, retains itself sequentially), sequential learning will let the model correct itself
    - Decision tree is apart of this

3. Stacking (put multiple models together), predictions from different types of models and average them, or weight each model and how much they contribute to each other, or build a meta model, outputs from the previous model will be meshed together and build together to make a new prediction

In [1]:
pip install sweetviz

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 21.6 MB/s  0:00:00 eta 0:00:01
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sweetviz]3/4 [sweetviz]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# importing libraries
import numpy as np
import pandas as pd 
import sweetviz as sv
from sklearn.model_selection import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# loading the datasets and initial EDA with SweetViz
insurance = pd.read_csv('/Users/diyapatel/Downloads/insurance.csv')
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
#visualize the data with SweetViz
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [5]:
# encoding categorical variables 
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [6]:
# prepare the data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

#splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

#scaling the features (don't need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# baseline linear regression model 
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# baseline random forest model 
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# print baseline results
print(f'Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest R2: {mse_rf:.2}, R2: {r2_rf:.2}')

Baseline Linear Regression MSE: 33566439.74, R2: 0.78
Baseline Random Forest R2: 2.2e+07, R2: 0.86


In [7]:
#CV with baseline models with r2 as the scoring metric 
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf}')

Baseline Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Baseline Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


In [8]:
# gridsearchCV for hyperparameter tuning of random forest 
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2,5],
    'max_features': ['log2', 'sqrt']
}

grid_search_rf = GridSearchCV(estimator=baseline_rf,
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)
print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')

Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV R2 Score: 0.84


In [9]:
# examine top 20 ranked parameter combinations from grid search
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
11           200        10                  5         log2         0.841202
3            200      None                  5         log2         0.840662
19           200        20                  5         log2         0.840662
10           100        10                  5         log2         0.840452
8            100        10                  2         log2         0.840383
9            200        10                  2         log2         0.839967
2            100      None                  5         log2         0.839195
18           100        20                  5         log2         0.839195
17           200        20                  2         log2         0.836639
1            200      None                  2         log2         0.836593
16           100        20                  2         log2         0.835721
0            100      None                  2         log2

From the top 20 GridSearchCV results, I would choose the model with:

- `n_estimators = 200`
- `max_depth = 10`
- `min_samples_split = 5`
- `max_features = 'log2'`

I would choose this combination because it has the highest mean cross-validation score (`0.841202`) among all tested models, so it performed best overall on unseen validation folds.

This choice also makes sense from a modeling perspective. Using `200` trees gives the random forest more stability than `100` trees, while `max_depth = 10` helps control overfitting compared with unlimited depth. Setting `min_samples_split = 5` adds additional regularization by preventing splits on very small sample groups, and `max_features = 'log2'` improves tree diversity, which often strengthens random forest performance. Overall, this parameter set provides the best balance of predictive accuracy and generalization based on the GridSearchCV results.


